## This notebook serves as a demonstration of the LASP SDTP application

#### Some commands to make things easier to test within this notebook

In [1]:
import json
from lasp_sdtp.database.database_controller import db
from tests import conftest

# Clear out contents of database
db.session.query(db.FileQueue).delete()
db.session.query(db.Transactions).delete()
db.session.query(db.TagsAndExtras).delete()
db.session.query(db.Files).delete()
db.session.query(db.MissionShortnameMapping).delete()
db.session.query(db.Shortnames).delete()
db.session.query(db.MissionAccountMapping).delete()
db.session.query(db.Missions).delete()
db.session.query(db.Accounts).delete()
db.session.commit()

# Add entries to database tables to support tests
conftest._add_missions_entries()
conftest._add_accounts_entries()
conftest._add_shortnames_entries()
conftest._add_mission_shortname_mapping_entries()
conftest._add_files_entries()

DatabaseError: (cx_Oracle.DatabaseError) ORA-00942: table or view does not exist
[SQL: DELETE FROM "FILEQUEUE"]
(Background on this error at: https://sqlalche.me/e/14/4xp6)

## Configuration

#### Subscribers must configure several parameters prior to using the `lasp_sdtp` application.  To configure a subscriber, instantiate the SubsciberConfig class, set the necessary parameters, and save.

In [ ]:
from lasp_sdtp.config import SubscriberConfig

In [ ]:
config = SubscriberConfig()

config.account_expiration_period = 1800  # The number of days the subscriber account will be valid
config.checksum_type = 'sha256'  # The type of checksum that the subscriber will use to validate responses
config.distinguished_name = ''  # The distingushed name to use in authentication
config.expiration_period = 180  # The number of days files will remain in the file queue
config.max_num_files = 10000  # The maximum number of files to return in file list requests
config.missions = ['TSIS2']
config.num_download_threads = 5  # The maximum number of simultaneous downloads
config.username = 'ges_disc'  # The subscriber username (used in authentication)

# Define the file stream(s) used for file transfer, along with any unique extras and tags
config.streams = {
    "prod": {
        "extras": {"type": {"type": "str", "default": ""}},
        "tags": {"date": {"type": "str", "default": ""}}
    },
    "dev": {
        "extras": {"was_extrapolated": {"type": "bool", "default": ""}},
        "tags": {"irradiance": {"type": "float", "default": ""}}
    }
}

In [ ]:
config.__dict__

In [ ]:
config.save()

## Register Account

Register a new account.  The user must supply the `Cert-UID` in the header of the request.  If successful, a `204` response is returned.
    
Command: `curl PUT "http://127.0.0.1:8000/register -H "Cert-UID: <certificate>"`

In [ ]:
! curl -i -X PUT "http://127.0.0.1:8000/register" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

## List Available Files

Return a list of available files given user-supplied parameters.  The user must supply the `Cert-UID` in the header of the request.  If successful, a `200` response is returned with a list of the available files and their metadata.  If no parameters are supplied, then all available files are returned.

Command: `curl GET "http://127.0.0.1:8000/files?param1=value1&param2=value2... -H "Cert-UID: <certificate>"`

#### Supported parameters:
- `stream` (e.g. `prod`)
- `ShortName` (e.g. `TSIS_SC_L2`)
- `version` (e.g. `01`)
- `date` (e.g.`2022-01-01`)
- `start_date` (e.g. `2022-01-01`)
- `end_date` (e.g. `2022-01-02`)


Note that `date` can only be supplied if `start_date` and/or `end_date` are *not* supplied.  Also, if `start_date` is supplied, then `end_date` *must* be supplied, or vice-versa 

#### Examples

In [ ]:
# List all available files
! curl -i -X GET "http://127.0.0.1:8000/files" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

In [ ]:
# List all available files with ShortName of 'TSIS_SC_L2'
! curl -i -X GET "http://127.0.0.1:8000/files?shortname=TSIS_SC_L2" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

In [ ]:
# List all available files with specific ShortName and for a specific stream
! curl -i -X GET "http://127.0.0.1:8000/files?shortname=TSIS_SC_L2&stream=dev" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

In [ ]:
# List all available files filtered by subscriber-specific tag
curl -i -X GET "http://127.0.0.1:8000/files?shortname=TSIS_SC_L2&observation_date=some_value" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

## Get Specific File

Return the contents of a file for a given `fileid`.  The user must supply the `Cert-UID` in the header of the request.  If successful, a `200` response is returned with the content of the file.  

Command: `curl GET "http://127.0.0.1:8000/files/<fileid> -H "Cert-UID: <certificate>"`

In [ ]:
! curl -i -X GET "http://127.0.0.1:8000/files/1810" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

## Delete Specific File

Remove a specific file from the queue.  The user must supply the `Cert-UID` in the header of the request.  If successful, a `204` response is returned with the content of the file.  

Command: `curl DELETE "http://127.0.0.1:8000/files/<fileid> -H "Cert-UID: <certificate>"`

In [ ]:
! curl -i -X DELETE "http://127.0.0.1:8000/files/1810" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

## Delete Range of Files

Remove a range of files from the queue.  The user must supply the `Cert-UID` in the header of the request.  The range of files should be positive integers that increase in value.  If successful, a `204` response is returned.

Command: `curl DELETE "http://127.0.0.1:8000/files/<fileid_start>-<fileid_end> -H "Cert-UID: <certificate>"`

In [ ]:
# First get a few files so that they are in the queue
! curl -i -X GET "http://127.0.0.1:8000/files/1810" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"
! curl -i -X GET "http://127.0.0.1:8000/files/1811" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"
! curl -i -X GET "http://127.0.0.1:8000/files/1812" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

In [ ]:
# Delete the range of files
! curl -i -X DELETE "http://127.0.0.1:8000/files/1810-1812" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

## Examples of bad/invalid requests

In [ ]:
# Send request to endpoint that doesn't exist
# Expect a 404 response
! curl -i -X GET "http://127.0.0.1:8000/bogus_endpoint" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

In [ ]:
# Try to register a closed account
# Expect a 401 response
! curl -i -X PUT "http://127.0.0.1:8000/register" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

In [ ]:
# Request a file list with bogus parameters
# Expect a 400 response
! curl -i -X GET "http://127.0.0.1:8000/files?bogus_param=foo" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X GET "http://127.0.0.1:8000/files?ShortName=TSIS_SC_L2&bogus_param=foo" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X GET "http://127.0.0.1:8000/files?date=20210101" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

In [ ]:
# Request a file list with valid parameters but no files returned
# Expect a 200 response with empty file list
! curl -i -X GET "http://127.0.0.1:8000/files?date=1984-04-04" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

In [ ]:
# Request a file that doesn't exist
# Expect a 404 response
! curl -i -X GET "http://127.0.0.1:8000/files/1234567" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

In [ ]:
# Request a file with invalid fileid
# Expect a 400 response
! curl -i -X GET "http://127.0.0.1:8000/files/-1" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X GET "http://127.0.0.1:8000/files/1.5" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X GET "http://127.0.0.1:8000/files/foo" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X GET "http://127.0.0.1:8000/files/9999999999999999" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

In [ ]:
# Try to delete a file that doesn't exist
# Expect a 404 response
! curl -i -X DELETE "http://127.0.0.1:8000/files/1234567" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

In [ ]:
# Try to delete a file with invalid fileid
# Expect a 400 response
! curl -i -X DELETE "http://127.0.0.1:8000/files/-1" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X DELETE "http://127.0.0.1:8000/files/1.5" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X DELETE "http://127.0.0.1:8000/files/foo" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

In [ ]:
# Try to delete a range of files that don't exist
# Expect a 404 response
! curl -i -X DELETE "http://127.0.0.1:8000/files/1234567-1234569" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

In [ ]:
# Try to delete a range of files with invalid fileids
# Expect a 400 response
! curl -i -X DELETE "http://127.0.0.1:8000/files/foo-bar" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X DELETE "http://127.0.0.1:8000/files/10-5" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X DELETE "http://127.0.0.1:8000/files/123-foo" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X DELETE "http://127.0.0.1:8000/files/123 - 456" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X DELETE "http://127.0.0.1:8000/files/10-10" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

In [ ]:
# Try to request a file that user doesn't have access to (file 23456 is hard-coded to be restricted)
# Expect a 403 response
! curl -i -X GET "http://127.0.0.1:8000/files/23456" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

In [ ]:
# Try to delete a file that user doesn't have access to (file 23456 is hard-coded to be restricted)
# Expect a 403 response
! curl -i -X DELETE "http://127.0.0.1:8000/files/23456" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

In [ ]:
# Send A LOT of requests at once